In [0]:
# O QUE: Validar as chaves da tabela Gold de vendas.
# POR QUÊ: Impedir registros sem identificação ou itens duplicados.

df = spark.table("workspace.gold.fato_vendas")

# Verificar identificadores nulos
nulos = df.filter(
    "order_id IS NULL OR order_item_id IS NULL"
).count()

# Verificar combinações duplicadas
duplicados = (
    df.groupBy("order_id", "order_item_id")
    .count()
    .filter("count > 1")
    .count()
)

# Interromper a execução se houver problemas
assert nulos == 0, f"Encontrados {nulos} registros com chaves nulas"
assert duplicados == 0, f"Encontradas {duplicados} chaves duplicadas"

print("Qualidade OK: nenhuma chave nula ou duplicada.")

In [0]:
# O QUE: Validar o relacionamento entre vendas e clientes.
# POR QUÊ: Identificar vendas sem cliente correspondente na Gold.

vendas = spark.table("workspace.gold.fato_vendas")
clientes = spark.table("workspace.gold.dim_cliente")

# Identificar clientes presentes nas vendas, mas ausentes na dimensão
vendas_sem_cliente = (
    vendas.select("customer_id").distinct()
    .join(
        clientes.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

# Interromper a execução se houver inconsistências
assert vendas_sem_cliente == 0, (
    f"Encontrados {vendas_sem_cliente} clientes sem correspondência"
)

print("Qualidade OK: todas as vendas possuem clientes válidos.")